# AID2026 — Incident Detection Training Notebook
Run each cell **in order**, top to bottom.

**First:** Set runtime to A100 → Runtime → Change runtime type → A100

## Cell 1 — Download Dataset from Shared Drive
Downloads all videos and CSVs directly from the competition shared folder into Colab. Takes 5–15 minutes.

In [ ]:
import os, zipfile, time
!pip install -q gdown
import gdown

# Create folder structure
os.makedirs('/content/aid2026/data/train', exist_ok=True)
os.makedirs('/content/aid2026/data/val',   exist_ok=True)

# --- CSVs (download in seconds) ---
print('Downloading train_GT.csv...')
gdown.download(id='1gktt4ZlS75ijx50ONbiFRkwWZkyb9b2l', output='/content/aid2026/data/train_GT.csv', quiet=False)

print('Downloading val_GT.csv...')
gdown.download(id='1h1f4aMyMH845t5yrcllglXFSs5E_lX_2', output='/content/aid2026/data/val_GT.csv', quiet=False)

# --- Train videos ---
print('\nDownloading train.zip...')
t0 = time.time()
gdown.download(id='1-ffvPj8aGUUUnb4tzpp8nrPK4Kl-nHUF', output='/content/aid2026/data/train.zip', quiet=False)
print(f'Downloaded in {(time.time()-t0)/60:.1f} mins. Unzipping...')
with zipfile.ZipFile('/content/aid2026/data/train.zip', 'r') as z:
    z.extractall('/content/aid2026/data/train/')
os.remove('/content/aid2026/data/train.zip')
print(f'Train videos: {len(os.listdir("/content/aid2026/data/train"))} files')

# --- Val videos ---
print('\nDownloading val.zip...')
t0 = time.time()
gdown.download(id='1KOAbJg1yL5AnK9nMN8qesNJeCQX-O0Gc', output='/content/aid2026/data/val.zip', quiet=False)
print(f'Downloaded in {(time.time()-t0)/60:.1f} mins. Unzipping...')
with zipfile.ZipFile('/content/aid2026/data/val.zip', 'r') as z:
    z.extractall('/content/aid2026/data/val/')
os.remove('/content/aid2026/data/val.zip')
print(f'Val videos: {len(os.listdir("/content/aid2026/data/val"))} files')

print('\nDataset ready!')

## Cell 2 — Upload Project Code
Upload your aid2026_code.zip from your PC using the file picker below.

In [ ]:
from google.colab import files
uploaded = files.upload()

## Cell 3 — Check GPU

In [173]:
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'GPU available   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name        : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM            : {vram:.1f} GB')
else:
    print('WARNING: No GPU — go to Runtime → Change runtime type → A100')

PyTorch version : 2.10.0+cu128
GPU available   : True
GPU name        : Tesla T4
VRAM            : 15.6 GB


## Cell 4 — Install Dependencies
Takes about 1 minute.

In [4]:
!pip install -q transformers==4.40.0 accelerate decord scikit-learn einops timm opencv-python-headless
print('All dependencies installed.')

All dependencies installed.


## Cell 5 — Verify Everything is in Place

In [174]:
import os, sys

PROJECT_DIR = '/content/aid2026'
DATA_DIR    = os.path.join(PROJECT_DIR, 'data')
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))
os.chdir(PROJECT_DIR)

expected = [
    'data/train_GT.csv',
    'data/val_GT.csv',
    'data/train',
    'data/val',
    'src/model.py',
    'src/dataset.py',
    'src/train.py',
    'test.py',
]

all_ok = True
for path in expected:
    exists = os.path.exists(os.path.join(PROJECT_DIR, path))
    icon = 'OK     ' if exists else 'MISSING'
    print(f'  [{icon}]  {path}')
    if not exists:
        all_ok = False

print()
print('Ready to train!' if all_ok else 'Fix missing files before continuing.')

  [OK     ]  data/train_GT.csv
  [OK     ]  data/val_GT.csv
  [OK     ]  data/train
  [OK     ]  data/val
  [OK     ]  src/model.py
  [OK     ]  src/dataset.py
  [OK     ]  src/train.py
  [OK     ]  test.py

Ready to train!


## Cell 6 — Sanity Check: Load One Video
Confirms video filenames in the CSV match the actual files on disk.

In [165]:
with open("data/train_GT.csv", "r", encoding="utf-8") as f:
    lines = f.readlines()

for i in range(355, 365):
    print(f"Line {i+1}: {lines[i]}")

Line 356: s7;00:22;00:00;00:07

Line 357: s8;00:03;00:00;00:03

Line 358: s9;00:18;00:04;00:10

Line 359: s11;00:18;00:07;"00:11

Line 360: "

Line 361: s12;00:12;00:04;00:08

Line 362: s13;00:13;00:04;00:07

Line 363: s14;00:11;00:04;00:07

Line 364: s15;00:23;00:11;00:15

Line 365: s16;00:22;00:07;00:16



In [175]:
import glob, pandas as pd
from dataset import read_video_frames

train_df = pd.read_csv('data/train_GT.csv', sep=";" ,on_bad_lines="skip", engine="python")
train_df.columns = train_df.columns.str.strip()
print(f'Train rows: {len(train_df)}  |  Columns: {list(train_df.columns)}')
print(f'Sample rows:')
print(train_df.head(3).to_string())

first_id = str(train_df.iloc[0]['Id Video']).strip()
print(f'\nTrying to load video ID: "{first_id}"')

candidates = (
    glob.glob(f'{DATA_DIR}/train/train/{first_id}') +
    glob.glob(f'{DATA_DIR}/train/train/{first_id}.*')
)

if candidates:
    frames, fps = read_video_frames(candidates[0], max_frames=32)
    print(f'SUCCESS: {len(frames)} frames at {fps:.1f} fps, shape {frames[0].shape}')
else:
    sample_files = glob.glob(f'{DATA_DIR}/train/train/*')[:5]
    print(f'ERROR: Video not found.')
    print(f'Sample filenames in data/train/train/: {[os.path.basename(f) for f in sample_files]}')
    print('The Id Video column values must match these filenames (with or without extension).')

Train rows: 1246  |  Columns: ['Id Video', 'Duration', 'Start', 'End']
Sample rows:
  Id Video Duration  Start    End
0       v1    00:23  00:15  00:20
1       v2    00:04  00:03  00:04
2       v3    00:14  00:05  00:09

Trying to load video ID: "v1"
SUCCESS: 32 frames at 59.5 fps, shape (1358, 2386, 3)


## Cell 7 — Configure Training
Safe to leave all settings as-is for the first run.

In [176]:
import json

cfg = {
    'backbone':        'MCG-NJU/videomae-small-finetuned-kinetics',
    'clip_frames':     16,
    'context_clips':   8,
    'dropout':         0.3,
    'freeze_layers':   8,
    'train_csv':       '/content/aid2026/data/train_GT.csv',
    'val_csv':         '/content/aid2026/data/val_GT.csv',
    'videos_dir':      '/content/aid2026/data/train',
    'clip_hop':        8,
    'max_clips':       48,
    'batch_size':      4,
    'num_workers':     2,
    'epochs':          30,
    'lr':              2e-4,
    'weight_decay':    1e-4,
    'patience':        8,
    'amp':             True,
    'checkpoint_dir':  f'{PROJECT_DIR}/checkpoints',
}

os.makedirs(f'{PROJECT_DIR}/configs', exist_ok=True)
cfg_path = f'{PROJECT_DIR}/configs/train_config.json'
with open(cfg_path, 'w') as f:
    json.dump(cfg, f, indent=2)

print('Config saved. Key settings:')
print(f'  Backbone : {cfg["backbone"]}')
print(f'  Epochs   : up to {cfg["epochs"]} (stops early if no improvement for {cfg["patience"]} epochs)')
print(f'  Batch    : {cfg["batch_size"]} videos per step')

Config saved. Key settings:
  Backbone : MCG-NJU/videomae-small-finetuned-kinetics
  Epochs   : up to 30 (stops early if no improvement for 8 epochs)
  Batch    : 4 videos per step


## Cell 8 — Train the Model
**Takes 2–4 hours. Do not close the tab.**

You will see output like this every 20 steps:
```
[E1 S0/415] loss=0.6821 lr=2.00e-04 t=12.3s
[Epoch 1] F1=0.6123 P=0.6500 R=0.5800 | delay=3.2s
[Checkpoint] Saved  (F1=0.6123)
```
The best model saves automatically whenever F1 improves.

In [21]:
!pip install -q huggingface_hub

In [26]:
from huggingface_hub import login
login()

In [181]:
%%writefile /content/aid2026/src/dataset.py
"""
AID2026 — Robust Dataset Loader (FINAL FIX)
- Safe CSV parsing
- Time format handling (MM:SS)
- Missing video tolerant
- Flexible filename matching
- No empty dataset crash
"""

import os
import csv
from pathlib import Path
from typing import List, Dict, Optional

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as T

try:
    from decord import VideoReader, cpu
    DECORD_AVAILABLE = True
except:
    DECORD_AVAILABLE = False
    import torchvision.io as tvio


# -------------------------
# CONFIG
# -------------------------
DEFAULT_CLIP_FRAMES = 16
DEFAULT_CLIP_STRIDE = 8
DEFAULT_CLIP_HOP = 8
DEFAULT_MAX_CLIPS = 64


# -------------------------
# TIME PARSER
# -------------------------
def time_to_seconds(t):
    if t is None:
        return 0.0
    t = str(t).strip()

    try:
        if ":" in t:
            parts = t.split(":")
            if len(parts) == 2:
                m, s = parts
                return int(m) * 60 + float(s)
            elif len(parts) == 3:
                h, m, s = parts
                return int(h)*3600 + int(m)*60 + float(s)
        return float(t)
    except:
        return 0.0


# -------------------------
# VIDEO LOADER
# -------------------------
def read_video_frames(path: str, max_frames: int = 4096):
    if DECORD_AVAILABLE:
        vr = VideoReader(path, ctx=cpu(0))
        fps = vr.get_avg_fps()
        n = min(len(vr), max_frames)
        frames = vr.get_batch(list(range(n))).asnumpy()
        return frames, fps
    else:
        video, _, info = tvio.read_video(path, pts_unit="sec")
        fps = info.get("video_fps", 25.0)
        return video.numpy()[:max_frames], fps


# -------------------------
# CLIP EXTRACTION
# -------------------------
def extract_clips(frames, clip_frames, clip_stride, clip_hop, max_clips):
    T = len(frames)

    starts = list(range(0, max(1, T - clip_frames * clip_stride + 1), clip_hop))
    if len(starts) == 0:
        starts = [0]

    clips = []
    for s in starts[:max_clips]:
        idx = [min(s + i * clip_stride, T - 1) for i in range(clip_frames)]
        clips.append(frames[idx])

    return np.stack(clips), starts[:max_clips]


# -------------------------
# DATASET
# -------------------------
class MIVIADataset(Dataset):

    def __init__(
        self,
        csv_path,
        videos_dir,
        train=True,
        clip_frames=DEFAULT_CLIP_FRAMES,
        clip_stride=DEFAULT_CLIP_STRIDE,
        clip_hop=DEFAULT_CLIP_HOP,
        max_clips=DEFAULT_MAX_CLIPS
    ):
        self.videos_dir = Path(videos_dir)
        self.train = train

        self.clip_frames = clip_frames
        self.clip_stride = clip_stride
        self.clip_hop = clip_hop
        self.max_clips = max_clips

        self.transform = T.Compose([
            T.ToDtype(torch.float32, scale=True),
            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])

        raw_samples = self._load_csv(csv_path)

        # FILTER VALID VIDEOS
        self.samples = []
        skipped = 0

        for s in raw_samples:
            if self._find_video(s["video_id"]) is not None:
                self.samples.append(s)
            else:
                skipped += 1

        print(f"[Dataset] Loaded {len(self.samples)} samples")
        print(f"[Dataset] Skipped {skipped} missing videos")

        if len(self.samples) == 0:
            raise ValueError(f"No valid samples found in {csv_path}")

    # -------------------------
    # CSV LOADER
    # -------------------------
    def _load_csv(self, path: str):
        samples = []

        with open(path, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f, delimiter=';')

            for row in reader:
                vid = str(row.get("Id Video", "")).strip()

                duration = time_to_seconds(row.get("Duration", "0"))
                start = row.get("Start", "")
                end = row.get("End", "")

                start = str(start).strip()
                end = str(end).strip()

                label = 1 if start != "" else 0

                samples.append({
                    "video_id": vid,
                    "duration": duration,
                    "label": label,
                    "onset_sec": time_to_seconds(start) if start else -1,
                    "end_sec": time_to_seconds(end) if end else -1,
                })

        return samples


    # -------------------------
    # VIDEO FINDER (ROBUST)
    # -------------------------
    def _find_video(self, vid: str) -> Optional[Path]:
        vid = str(vid).strip()

        if vid == "":
            return None

        folders = [
            self.videos_dir,
            self.videos_dir / "train",
            self.videos_dir / "val"
        ]

        exts = ["", ".mp4", ".avi", ".mov"]

        for folder in folders:
            if not folder.exists():
                continue

            for ext in exts:
                p = folder / f"{vid}{ext}"
                if p.is_file():
                    return p

        return None


    # -------------------------
    # LENGTH
    # -------------------------
    def __len__(self):
        return len(self.samples)


    # -------------------------
    # GET ITEM
    # -------------------------
    def __getitem__(self, idx):
        s = self.samples[idx]

        vid_path = self._find_video(s["video_id"])
        frames, fps = read_video_frames(str(vid_path))

        clips, starts = extract_clips(
            frames,
            self.clip_frames,
            self.clip_stride,
            self.clip_hop,
            self.max_clips
        )

        clips = torch.from_numpy(clips).permute(0,1,4,2,3).float()

        return {
            "clips": clips,
            "label": torch.tensor(s["label"]),
            "video_id": s["video_id"]
        }


# -------------------------
# DATALOADER
# -------------------------
def build_dataloaders(
    train_csv,
    val_csv,
    videos_dir,
    batch_size=4,
    num_workers=2,
    **kwargs
):

    train_ds = MIVIADataset(train_csv, videos_dir, train=True, **kwargs)
    val_ds = MIVIADataset(val_csv, videos_dir, train=False, **kwargs)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    return train_loader, val_loader

Overwriting /content/aid2026/src/dataset.py


In [78]:
%%writefile -a /content/aid2026/src/dataset.py

def build_dataloaders(
    train_csv,
    val_csv,
    videos_dir,
    batch_size=4,
    num_workers=2,
    **dataset_kwargs
):
    train_ds = MIVIADataset(train_csv, videos_dir, train=True, **dataset_kwargs)
    val_ds   = MIVIADataset(val_csv, videos_dir, train=False, **dataset_kwargs)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    return train_loader, val_loader

Appending to /content/aid2026/src/dataset.py


In [182]:
!grep -n "build_dataloaders" /content/aid2026/src/dataset.py

236:def build_dataloaders(


In [183]:
!python {PROJECT_DIR}/src/train.py --config {cfg_path}

[Trainer] Device: cuda
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
[Trainer] Trainable parameters: 8,088,450
[Dataset] Loaded 0 samples
[Dataset] Skipped 1246 missing videos
Traceback (most recent call last):
  File "/content/aid2026/src/train.py", line 378, in <module>
    trainer = Trainer(cfg)
              ^^^^^^^^^^^^
  File "/content/aid2026/src/train.py", line 156, in __init__
    self.train_loader, self.val_loader = build_dataloaders(
                                         ^^^^^^^^^^^^^^^^^^
  File "/content/aid2026/src/dataset.py", line 245, in build_dataloaders
    train_ds = MIVIADataset(train_csv, videos_dir, train=True, **kwargs)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/aid2026/src/datase

In [180]:
import pandas as pd

df = pd.read_csv("/content/aid2026/data/train_GT.csv", sep=";")
print(df.head())
print(df.columns)
print(len(df))

  Id Video Duration  Start    End
0       v1    00:23  00:15  00:20
1       v2    00:04  00:03  00:04
2       v3    00:14  00:05  00:09
3       v4    00:09  00:02  00:06
4       v5    00:11  00:04  00:06
Index(['Id Video', 'Duration', 'Start', 'End'], dtype='object')
1246


## Cell 9 — Plot Training Curves
Run after training finishes to see how F1 improved over time.

In [ ]:
import json, matplotlib.pyplot as plt

epochs, f1s, precs, recs, delays = [], [], [], [], []
with open(f'{PROJECT_DIR}/checkpoints/metrics_log.jsonl') as f:
    for line in f:
        m = json.loads(line)
        epochs.append(m['epoch']); f1s.append(m['f1'])
        precs.append(m['precision']); recs.append(m['recall'])
        delays.append(m['avg_delay_sec'])

best_idx = f1s.index(max(f1s))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs, f1s,   label='F1',       linewidth=2, color='green')
axes[0].plot(epochs, precs, label='Precision', linewidth=2, color='blue',   linestyle='--')
axes[0].plot(epochs, recs,  label='Recall',    linewidth=2, color='orange', linestyle=':')
axes[0].axvline(x=epochs[best_idx], color='green', alpha=0.3, label=f'Best epoch ({epochs[best_idx]})')
axes[0].set_title('Validation Metrics'); axes[0].set_ylim(0,1); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(epochs, delays, color='red', linewidth=2)
axes[1].set_title('Avg Notification Delay (s)'); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Best F1    : {f1s[best_idx]:.4f}  (epoch {epochs[best_idx]})')
print(f'Precision  : {precs[best_idx]:.4f}')
print(f'Recall     : {recs[best_idx]:.4f}')
print(f'Avg Delay  : {delays[best_idx]:.2f}s')

## Cell 10 — Save Checkpoint to Google Drive ⚠️
**Run before closing Colab.** Saves your trained model to Drive so you don't lose it.

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/AID2026/checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

ckpt = f'{PROJECT_DIR}/checkpoints/checkpoint_best.pt'
if os.path.exists(ckpt):
    shutil.copy(ckpt, f'{SAVE_DIR}/checkpoint_best.pt')
    shutil.copy(f'{PROJECT_DIR}/checkpoints/metrics_log.jsonl', f'{SAVE_DIR}/metrics_log.jsonl')
    print(f'Saved to Drive: {SAVE_DIR}')
else:
    print('Checkpoint not found. Has Cell 8 finished?')

## Cell 11 — Package & Save Submission to Drive

In [ ]:
import shutil, glob
from google.colab import drive
drive.mount('/content/drive')

SUB = '/content/submission'
if os.path.exists(SUB): shutil.rmtree(SUB)
os.makedirs(SUB)

shutil.copy(f'{PROJECT_DIR}/test.py', f'{SUB}/test.py')
shutil.copytree(f'{PROJECT_DIR}/src', f'{SUB}/src')
shutil.copytree(f'{PROJECT_DIR}/checkpoints', f'{SUB}/checkpoints')
nb = glob.glob('/content/*.ipynb')
if nb: shutil.copy(nb[0], f'{SUB}/test.ipynb')

zip_path = '/content/drive/MyDrive/AID2026/aid2026_submission'
shutil.make_archive(zip_path, 'zip', SUB)
size_mb = os.path.getsize(zip_path + '.zip') / 1e6
print(f'Submission saved to Drive ({size_mb:.0f} MB)')
print(f'Location: {zip_path}.zip')